In [ ]:
import logging

import torch
from datasets import load_dataset

from bugulma_enjoyers.detoxifiers import PipelineConfig, StandaloneDetoxifier, BacktranslationDetoxifier


In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
toxic = load_dataset("textdetox/multilingual_toxic_lexicon")
ds = load_dataset("textdetox/multilingual_paradetox_test")

In [ ]:
toxic_words = []
for word in toxic["ru"]["text"]:
    toxic_words.append(word)

In [ ]:
config = PipelineConfig(batch_size=4, device="cpu", detoxifier_model_name="yandex/yandexgpt/rc")
detox = StandaloneDetoxifier(config)

INFO:bugulma_enjoyers.detoxifiers.standalone:Loading Model: yandex/yandexgpt/rc


In [ ]:
text_results = []
lang = "tt"
for text in ds["tt"]["text"]:
    logger.info("Initial text: %s", text)
    result = detox.detoxify(text, lang)
    text_results.append(result)
    logger.info("Result: %s", result)
    break

INFO:__main__:Initial text: @user, ну чапай чапай эйтер идем инде;-)эйтеп тормыйм
ERROR:bugulma_enjoyers.models._api_model:Error parsing model response
Traceback (most recent call last):
  File "c:\Users\user\Desktop\hack\benj\BugulmaEnjoyers\bugulma_enjoyers\models\_api_model.py", line 119, in forward
    json_str = self.clean_json_response(text_response)
               ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'YandexModel' object has no attribute 'clean_json_response'
INFO:__main__:Result: @user, ну чапай чапай эйтер идем инде;-)эйтеп тормыйм


In [ ]:
config = PipelineConfig(batch_size=4, device="cpu")
detox = BacktranslationDetoxifier(config, detox)

INFO:bugulma_enjoyers.detoxifiers.backtranslation:Loading translator model: hf/facebook/nllb-200-distilled-600M


In [ ]:
text_results = []
lang = "tt"
for text in ds["tt"]["text"]:
    logger.info("Initial text: %s", text)
    result = detox.detoxify(text, lang)
    text_results.append(result)
    logger.info("Result: %s", result)
    break

INFO:__main__:Initial text: @user, ну чапай чапай эйтер идем инде;-)эйтеп тормыйм
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
ERROR:bugulma_enjoyers.models._api_model:Error parsing model response
Traceback (most recent call last):
  File "c:\Users\user\Desktop\hack\benj\BugulmaEnjoyers\bugulma_enjoyers\models\_api_model.py", line 119, in forward
    json_str = self.clean_json_response(text_response)
               ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'YandexModel' object has no attribute 'clean_json_response'
INFO:__main__:Result: @user, мин инде күп ашаган булыр идем;-)


In [ ]:
detox.detoxify("Go fuck yourself", "en")

ERROR:bugulma_enjoyers.models._api_model:Error parsing model response
Traceback (most recent call last):
  File "c:\Users\user\Desktop\hack\benj\BugulmaEnjoyers\bugulma_enjoyers\models\_api_model.py", line 119, in forward
    json_str = self.clean_json_response(text_response)
               ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'YandexModel' object has no attribute 'clean_json_response'


'Go fuck yourself .'

In [ ]:
import pandas as pd

df = pd.read_csv("ttt.tsv", sep="\t")

### Open Router test

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [1]:
from openai import OpenAI
import os
api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

print(api_key)

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


In [4]:
completion = client.chat.completions.create(
  extra_headers={
    "HTTP-Referer": "<YOUR_SITE_URL>", # Optional. Site URL for rankings on openrouter.ai.
    "X-Title": "<YOUR_SITE_NAME>", # Optional. Site title for rankings on openrouter.ai.
  },
  extra_body={},
  model="google/gemini-2.5-pro",
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "What is in this image, audio and video?"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://live.staticflickr.com/3851/14825276609_098cac593d_b.jpg"
          }
        },
        {
          "type": "video_url",
          "video_url": {
            "url": "https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/1080/Big_Buck_Bunny_1080_10s_5MB.mp4"
          }
        }
      ]
    }
  ]
)
print(completion.choices[0].message.content)

Based on the media provided, here is a breakdown of the contents of the image, audio, and video.

### **Image Content**

The image captures two dolphins in the middle of a deep blue ocean.

*   **Main Subject:** The focus is on two Common Dolphins (*Delphinus delphis*), identifiable by their distinctive coloring which includes dark gray backs, yellowish-tan forward flanks, and lighter gray rear flanks creating a crisscross or hourglass pattern.
*   **Action:** One dolphin is in the foreground, leaping out of the water in a dynamic pose, creating a white splash. Its sleek body is almost entirely visible. To its left and slightly behind, a second dolphin is partially submerged, with its dorsal fin and upper back breaking the surface.
*   **Environment:** The dolphins are surrounded by the rich blue water of the ocean, which is textured with ripples and small waves, reflecting the daylight.

### **Audio Content**

The provided video clip is **silent**. There is no accompanying audio, musi

In [ ]:
import requests

def check_openrouter_balance(api_key):
    url = "https://openrouter.ai/api/v1/auth/key"
    headers = {
        "Authorization": f"Bearer {api_key}"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()['data']
        
        print(f"--- Информация о ключе ---")
        print(f"Название (Label): {data.get('label', 'Не указано')}")
        
        usage = data.get('usage', 0)
        limit = data.get('limit')
        
        print(f"Потрачено (Usage): ${usage:.4f}")
        
        if limit:
            print(f"Лимит ключа: ${limit:.4f}")
            remaining = limit - usage
            print(f"Осталось (при наличии лимита): ${remaining:.4f}")
        else:
            print("Лимит на ключе не установлен (используется общий баланс аккаунта).")
            print(f"Если бюджет $5.00, то осталось примерно: ${5.00 - usage:.4f}")
            
    except Exception as e:
        print(f"Ошибка при проверке: {e}")

check_openrouter_balance(api_key=api_key)

--- Информация о ключе ---
Название (Label): sk-or-v1-c3a...a92
Потрачено (Usage): $0.0123
Лимит ключа: $5.0000
Осталось (при наличии лимита): $4.9877
